# Máy chủ suy diễn và đo lường hiệu năng CatVTON trên GPU (FastAPI + ngrok)

Notebook triển khai dịch vụ thử đồ ảo trên môi trường Kaggle GPU (`NVIDIA Tesla T4`), bao gồm 3 ô thực thi tuần tự:
1. **Thiết lập môi trường**: Tải mã nguồn `CatVTON` và cài đặt các gói phụ thuộc tương thích với nhân `PyTorch`/`CUDA` có sẵn trên Kaggle.
2. **Khởi tạo pipeline suy diễn**: Nạp trọng số `CatVTON` và `Stable Diffusion Inpainting` ở chế độ `FP16`, kích hoạt bộ lập lịch `DPM++ 2M Karras`, khởi tạo `AutoMasker` (`DensePose` + `SCHP`) và ghi nhận thời gian khởi động nguội (`COLD_START_TIME_S`).
3. **Khởi chạy máy chủ API**: Cung cấp hai điểm cuối `/api/health` và `/api/tryon`, đồng bộ hóa hàng đợi `CUDA` bằng `torch.cuda.synchronize()` để trả về thông số phân rã độ trễ (`preprocess`, `inference`, `postprocess`, `peak VRAM`) qua HTTP response headers và mở đường hầm `ngrok`.

Cấu hình khóa xác thực thông qua Kaggle Secrets (`Add-ons -> Secrets`) hoặc biến môi trường trước khi chạy Cell 3:
- `NGROK_AUTHTOKEN`: Mã xác thực tài khoản `ngrok` (bắt buộc).
- `NGROK_DOMAIN`: Tên miền tĩnh `ngrok` (tùy chọn, ví dụ: `aspire-traps-deserving.ngrok-free.dev`).

In [ ]:
# Cell 1: Thiet lap moi truong va cai dat thu vien phu thuoc tren Kaggle
import os

os.chdir("/kaggle/working")
if not os.path.isdir("CatVTON"):
    !git clone https://github.com/Zheng-Chong/CatVTON.git
%cd /kaggle/working/CatVTON

# Loai bo cac goi ghim phien ban gay xung dot voi PyTorch/CUDA cai san tren Kaggle
!grep -viE '^(gradio|huggingface_hub|torch|torchvision|accelerate|diffusers|peft)' requirements.txt > requirements_notebook.txt
!pip install -q -r requirements_notebook.txt
!pip install -q "diffusers>=0.30.0" "accelerate>=0.33.0" "transformers>=4.44.0" "peft>=0.13.0" "huggingface_hub>=0.34.0,<2.0" fvcore iopath yacs pycocotools omegaconf cloudpickle av
!pip install -q fastapi uvicorn python-multipart pyngrok nest-asyncio
print("[OK] Thiet lap moi truong hoan tat.")

In [ ]:
# Cell 2: Nap checkpoint, khoi tao CatVTON FP16 + AutoMasker va do thoi gian cold-start
import glob
import io
import os
import time
import torch
from PIL import Image
from diffusers import DPMSolverMultistepScheduler
from diffusers.image_processor import VaeImageProcessor
from huggingface_hub import snapshot_download
from model.cloth_masker import AutoMasker
from model.pipeline import CatVTONPipeline
from utils import init_weight_dtype, resize_and_crop, resize_and_padding

torch.backends.cudnn.benchmark = True
t_cold_start_begin = time.perf_counter()


def find_local_dir(root, folder_name):
    """Tim thu muc folder_name ben trong root, tra ve duong dan neu ton tai va co tep."""
    if not os.path.isdir(root):
        return None
    for match in glob.glob(os.path.join(root, "**", folder_name), recursive=True):
        if os.path.isdir(match) and os.listdir(match):
            return match
    return None


DATASET_ROOT = "/kaggle/input/catvton-weights"
WORKING_ROOT = "/kaggle/working/catvton_weights"

catvton_local = find_local_dir(DATASET_ROOT, "CatVTON") or os.path.join(WORKING_ROOT, "CatVTON")
if os.path.isdir(catvton_local) and os.listdir(catvton_local):
    repo_path = catvton_local
    print(f"[CACHE] Su dung CatVTON weights co san: {repo_path}")
else:
    print("[DOWNLOAD] Tai CatVTON weights tu Hugging Face Hub...")
    repo_path = snapshot_download(repo_id="zhengchong/CatVTON", local_dir=catvton_local)

sd_local = find_local_dir(DATASET_ROOT, "sd-inpainting") or os.path.join(WORKING_ROOT, "sd-inpainting")
if os.path.isdir(sd_local) and os.listdir(sd_local):
    base_ckpt_path = sd_local
    print(f"[CACHE] Su dung SD-Inpainting weights co san: {base_ckpt_path}")
else:
    print("[DOWNLOAD] Tai SD-Inpainting weights tu Hugging Face Hub...")
    base_ckpt_path = snapshot_download(
        repo_id="booksforcharlie/stable-diffusion-inpainting",
        local_dir=sd_local,
    )

pipeline = CatVTONPipeline(
    base_ckpt=base_ckpt_path,
    attn_ckpt=repo_path,
    attn_ckpt_version="mix",
    weight_dtype=init_weight_dtype("fp16"),
    use_tf32=True,
    device="cuda",
    skip_safety_check=True,
)
mask_processor = VaeImageProcessor(
    vae_scale_factor=8,
    do_normalize=False,
    do_binarize=True,
    do_convert_grayscale=True,
)
automasker = AutoMasker(
    densepose_ckpt=os.path.join(repo_path, "DensePose"),
    schp_ckpt=os.path.join(repo_path, "SCHP"),
    device="cuda",
)

# Cau hinh bo lap lich DPM++ 2M Karras (20 buoc lay mau)
pipeline.noise_scheduler = DPMSolverMultistepScheduler.from_config(
    pipeline.noise_scheduler.config,
    use_karras_sigmas=True,
    algorithm_type="dpmsolver++",
)
print("[OPT] Scheduler: DPM++ 2M Karras - OK")

# Kich hoat xFormers memory-efficient attention neu moi truong ho tro
try:
    pipeline.unet.enable_xformers_memory_efficient_attention()
    print("[OPT] xFormers memory-efficient attention - OK")
except Exception as exc:
    print(f"[INFO] Su dung PyTorch SDPA mac dinh thay cho xFormers: {exc}")

torch.cuda.synchronize()
COLD_START_TIME_S = round(time.perf_counter() - t_cold_start_begin, 3)
INITIAL_VRAM_GB = round(torch.cuda.memory_allocated() / (1024 ** 3), 3)
print(
    f"[PROFILING] Cold-start hoan tat trong: {COLD_START_TIME_S}s | "
    f"VRAM khoi tao: {INITIAL_VRAM_GB} GB"
)

In [ ]:
# Cell 3: Khoi chay FastAPI Profiling Server va ket noi duong ham ngrok
import nest_asyncio
import uvicorn
from fastapi import FastAPI, File, Form, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pyngrok import ngrok

nest_asyncio.apply()

app = FastAPI(title="CatVTON Profiled Inference Server")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
    expose_headers=["*"],
)

OUTPUT_DIR = "/kaggle/working/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def get_runtime_secret(secret_name: str, default_value: str = "") -> str:
    """Doc cau hinh tu Kaggle Secrets (neu co) hoac bien moi truong."""
    try:
        from kaggle_secrets import UserSecretsClient
        val = UserSecretsClient().get_secret(secret_name)
        if val:
            return val.strip()
    except Exception:
        pass
    return os.environ.get(secret_name, default_value).strip()


@app.get("/api/health")
async def health_check():
    return {
        "status": "HEALTHY",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None",
        "cold_start_time_s": COLD_START_TIME_S,
        "current_vram_gb": round(torch.cuda.memory_allocated() / (1024 ** 3), 3),
        "precision": "fp16",
    }


@app.post("/api/tryon")
async def tryon_api(
    person_image: UploadFile = File(...),
    cloth_image: UploadFile = File(...),
    category: str = Form("overall"),
):
    t_req_start = time.perf_counter()

    person_bytes = await person_image.read()
    cloth_bytes = await cloth_image.read()
    person_img = Image.open(io.BytesIO(person_bytes)).convert("RGB")
    cloth_img = Image.open(io.BytesIO(cloth_bytes)).convert("RGB")

    width, height = 768, 1024

    # 1. Tien xu ly (Resize/Crop + AutoMasker DensePose/SCHP + Mask Blur)
    torch.cuda.synchronize()
    t_pre_start = time.perf_counter()
    person_img = resize_and_crop(person_img, (width, height))
    cloth_img = resize_and_padding(cloth_img, (width, height))
    mask = automasker(person_img, category)["mask"]
    mask = mask_processor.blur(mask, blur_factor=9)
    torch.cuda.synchronize()
    preprocess_time_s = round(time.perf_counter() - t_pre_start, 3)

    # 2. Suy dien khuach tan (20 buoc DPM++ 2M Karras)
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t_infer_start = time.perf_counter()
    with torch.inference_mode():
        result = pipeline(
            image=person_img,
            condition_image=cloth_img,
            mask=mask,
            num_inference_steps=20,
            guidance_scale=4.5,
        )[0]
    torch.cuda.synchronize()
    inference_time_s = round(time.perf_counter() - t_infer_start, 3)

    # 3. Hau xu ly va luu tep ket qua PNG
    t_post_start = time.perf_counter()
    ts_now = int(time.time() * 1000)
    out_path = os.path.join(OUTPUT_DIR, f"result_{ts_now}.png")
    result.save(out_path)
    postprocess_time_s = round(time.perf_counter() - t_post_start, 3)

    internal_total_latency_s = round(time.perf_counter() - t_req_start, 3)
    peak_vram_gb = round(torch.cuda.max_memory_allocated() / (1024 ** 3), 3)

    print(
        f"[PROFILING] Preprocess: {preprocess_time_s}s | "
        f"Inference: {inference_time_s}s | "
        f"Postprocess: {postprocess_time_s}s | "
        f"Total: {internal_total_latency_s}s | "
        f"Peak VRAM: {peak_vram_gb} GB",
        flush=True,
    )

    headers = {
        "X-Cold-Start-Sec": str(COLD_START_TIME_S),
        "X-Preprocess-Time-Sec": str(preprocess_time_s),
        "X-Inference-Time-Sec": str(inference_time_s),
        "X-Postprocess-Time-Sec": str(postprocess_time_s),
        "X-Internal-Total-Sec": str(internal_total_latency_s),
        "X-Peak-VRAM-GB": str(peak_vram_gb),
        "X-Weight-Precision": "fp16",
        "X-Inference-Steps": "20",
        "Access-Control-Expose-Headers": (
            "X-Cold-Start-Sec, X-Preprocess-Time-Sec, X-Inference-Time-Sec, "
            "X-Postprocess-Time-Sec, X-Internal-Total-Sec, X-Peak-VRAM-GB, "
            "X-Weight-Precision, X-Inference-Steps"
        ),
    }
    return FileResponse(out_path, media_type="image/png", headers=headers)


# Giai phong tien trinh ngrok cu va dat auth token truoc khi mo tunnel (tranh ERR_NGROK_4018)
try:
    ngrok.kill()
except Exception:
    pass

NGROK_TOKEN = get_runtime_secret("NGROK_AUTHTOKEN", "<YOUR_NGROK_AUTHTOKEN>")
NGROK_DOMAIN = get_runtime_secret("NGROK_DOMAIN", "")
ngrok.set_auth_token(NGROK_TOKEN)

if NGROK_DOMAIN:
    public_url = ngrok.connect(8000, domain=NGROK_DOMAIN).public_url
else:
    public_url = ngrok.connect(8000).public_url

print(f"API SẴN SÀNG TẠI: {public_url}/api/tryon")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()